# Model Industrialization

## Model Building

### Model Training

In [1]:
import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error

In [2]:
# load training data and split right away before any preprocessing
df = pd.read_csv('../data/train.csv')

continuous_features = ['GrLivArea', 'TotalBsmtSF']
categorical_features = ['Neighborhood', 'BldgType']
target = 'SalePrice'

X = df[continuous_features + categorical_features].copy()
y = df[target].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train size: {X_train.shape}')
print(f'Test size: {X_test.shape}')

Train size: (1168, 4)
Test size: (292, 4)


In [3]:
# fill missing values using train set statistics
for col in continuous_features:
    median_val = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_val)
    X_test[col] = X_test[col].fillna(median_val)

for col in categorical_features:
    mode_val = X_train[col].mode()[0]
    X_train[col] = X_train[col].fillna(mode_val)
    X_test[col] = X_test[col].fillna(mode_val)

print('Missing values after handling:')
print(X_train.isnull().sum())

Missing values after handling:
GrLivArea       0
TotalBsmtSF     0
Neighborhood    0
BldgType        0
dtype: int64


In [4]:
# scale continuous features - fit on train only, then transform
scaler = StandardScaler()
scaler.fit(X_train[continuous_features])
X_train_cont = scaler.transform(X_train[continuous_features])
X_test_cont = scaler.transform(X_test[continuous_features])

# save scaler for inference
joblib.dump(scaler, '../models/scaler.joblib')
print('Scaler saved')

Scaler saved


In [5]:
# encode categorical features - fit on train only, then transform
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(X_train[categorical_features])
X_train_cat = encoder.transform(X_train[categorical_features])
X_test_cat = encoder.transform(X_test[categorical_features])

# save encoder for inference
joblib.dump(encoder, '../models/encoder.joblib')
print('Encoder saved')

Encoder saved


In [6]:
# combine features
X_train_processed = np.hstack([X_train_cont, X_train_cat])
X_test_processed = np.hstack([X_test_cont, X_test_cat])
print(f'Processed train shape: {X_train_processed.shape}')
print(f'Processed test shape: {X_test_processed.shape}')

Processed train shape: (1168, 32)
Processed test shape: (292, 32)


In [7]:
# train the model and save it
model = LinearRegression()
model.fit(X_train_processed, y_train)
joblib.dump(model, '../models/model.joblib')
print('Model trained and saved')

Model trained and saved


### Model Evaluation

In [8]:
# load model from disk and evaluate on test set
model = joblib.load('../models/model.joblib')

y_pred = model.predict(X_test_processed)
y_pred = np.clip(y_pred, 1, None)

rmsle = np.sqrt(mean_squared_log_error(y_test, y_pred))
print(f'Test RMSLE: {round(rmsle, 2)}')

Test RMSLE: 0.2


## Model Inference

In [9]:
# load test data for inference
test_df = pd.read_csv('../data/test.csv')
X_inference = test_df[continuous_features + categorical_features].copy()
print(f'Inference data shape: {X_inference.shape}')
X_inference.head()

Inference data shape: (1459, 4)


,GrLivArea,TotalBsmtSF,Neighborhood,BldgType
0,896,882.0,NAmes,1Fam
1,1329,1329.0,NAmes,1Fam
2,1629,928.0,Gilbert,1Fam
3,1604,926.0,Gilbert,1Fam
4,1280,1280.0,StoneBr,TwnhsE


In [10]:
# fill missing values
for col in continuous_features:
    X_inference[col] = X_inference[col].fillna(X_inference[col].median())
for col in categorical_features:
    X_inference[col] = X_inference[col].fillna(X_inference[col].mode()[0])

In [11]:
# load saved scaler and encoder from disk
scaler = joblib.load('../models/scaler.joblib')
encoder = joblib.load('../models/encoder.joblib')

X_inf_cont = scaler.transform(X_inference[continuous_features])
X_inf_cat = encoder.transform(X_inference[categorical_features])
X_inf_processed = np.hstack([X_inf_cont, X_inf_cat])
print(f'Processed inference shape: {X_inf_processed.shape}')

Processed inference shape: (1459, 32)


In [12]:
# load saved model and make predictions
model = joblib.load('../models/model.joblib')
predictions = model.predict(X_inf_processed)
predictions = np.clip(predictions, 1, None)
print(f'Predictions shape: {predictions.shape}')
print(predictions[:10])

Predictions shape: (1459,)
[117704.93616453 159898.29933826 195798.32808388 194134.82658917
 247559.56850071 192106.4472695  175323.34432612 180800.81940022
 189454.25586721 116809.72050975]
